# Atlas-WX: run everything (one cell)
**Runtime → Change runtime type → A100 GPU, High RAM → Save.** Then run cell A and click **"Connect to Google Drive"** when asked. It fetches the 8,791-station data (~100 min, only the first time), copies it to Drive (`PWM-WX/isd`), then trains the three Stage 1 models (10M, then 28.5M and 97.6M grown from it) into `PWM-WX/runs`. It runs in the background: you can close the tab. If the runtime ends, open this notebook and run cell A again; every step resumes where it stopped.

In [ ]:
# A. Run everything
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/pwm || git clone -q -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model /content/pwm
!cd /content/pwm && git pull -q && git log --oneline -1
!nohup bash /content/pwm/atlas/wx/stage1_pipeline.sh < /dev/null > /content/pipeline.out 2>&1 &
!sleep 20; tail -3 /content/drive/MyDrive/PWM-WX/runs/pipeline.log; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; free -g | head -2

In [ ]:
# B. Progress (any time)
!tail -5 /content/drive/MyDrive/PWM-WX/runs/pipeline.log; tail -2 /content/drive/MyDrive/PWM-WX/runs/fetch.log 2>/dev/null | cut -c1-200; tail -3 /content/drive/MyDrive/PWM-WX/runs/log_wx_s1_*.txt 2>/dev/null

---
## Separate steps (only if needed)

# Atlas-WX data: every qualifying GHCNh station, 1995–2025
Runs `atlas/wx/fetch_ghcnh_all.py` (rules: `atlas/PREREG_WX.md` §1 and its addenda; GHCNh replaced ISD-Lite because NOAA stopped updating ISD in August 2025). Raw files and the packed output go to the runtime's local disk (`/content/wx_out`); cell 4 copies the output to Drive (`PWM-WX/isd`). No account or key is needed for the data.

Order: cell 1 (code), cell 2 (200-station trial, ~3 min), cell 3 (full fetch in the background, ~2 h), cell 4 (Drive: mount and copy; needs your click on "Connect to Google Drive").

In [ ]:
# 1. Code
!test -d /content/pwm || git clone -q -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model /content/pwm
!cd /content/pwm && git pull -q && git log --oneline -1 && df -h /content | tail -1 && free -g | head -2

In [ ]:
# 2. Trial on the first 200 candidate stations
!cd /content/pwm/atlas/wx && python -u fetch_ghcnh_all.py --raw /content/gh_raw --out /content/wx_trial --limit 200 2>&1 | tail -3

In [ ]:
# 3. Full fetch in the background (stdin from /dev/null so the cell returns at once; resumable: run again after a disconnect)
!mkdir -p /content/wx_out
!cd /content/pwm/atlas/wx && nohup python -u fetch_ghcnh_all.py --raw /content/gh_raw --out /content/wx_out < /dev/null > /content/wx_out/fetch.log 2>&1 &
!sleep 5; tail -2 /content/wx_out/fetch.log

In [ ]:
# 4. After the log ends with a WX-DATA line: mount Drive and copy the packed data (a few GB)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/PWM-WX/isd && rsync -a --info=progress2 --exclude fetch.log /content/wx_out/ /content/drive/MyDrive/PWM-WX/isd/ && cp /content/wx_out/fetch.log /content/drive/MyDrive/PWM-WX/isd/
!ls -la /content/drive/MyDrive/PWM-WX/isd | head; cat /content/drive/MyDrive/PWM-WX/isd/summary.json | head -c 1500

In [ ]:
# Progress (any time)
!tail -3 /content/wx_out/fetch.log; du -sh /content/gh_raw 2>/dev/null

## Stage 1 (after the data step has finished; a GPU runtime with high RAM, ideally A100)
Each cell is resumable: if the session ends, run the same cell again. Order: 10M, then 30M and 100M grown from it. Results go to `PWM-WX/runs`.

In [ ]:
# 5. Stage 1, 10M
!cd /content/pwm && git pull -q
!mkdir -p /content/drive/MyDrive/PWM-WX/runs
!cd /content/pwm/atlas/wx && nohup python -u run_atlas_wx.py --size 10000000 --tag wx_s1_10m --pause-at-cap --cap 600 > /content/drive/MyDrive/PWM-WX/runs/log_wx_s1_10m.txt 2>&1 &

In [ ]:
# 6. Stage 1, 30M and 100M (grown from the trained 10M; run after cell 5 has written wx_s1_10m_s600.json)
R='/content/drive/MyDrive/PWM-WX/runs'
!cd /content/pwm/atlas/wx && nohup sh -c "python -u run_atlas_wx.py --init $R/wx_s1_10m_s600.pt --grow 1,7,7 --lr 3e-4 --tag wx_s1_30m --pause-at-cap --cap 600 > $R/log_wx_s1_30m.txt 2>&1; python -u run_atlas_wx.py --init $R/wx_s1_10m_s600.pt --grow 2,6,6 --lr 3e-4 --tag wx_s1_100m --pause-at-cap --cap 600 > $R/log_wx_s1_100m.txt 2>&1" &

In [ ]:
# 7. Progress
!tail -3 /content/drive/MyDrive/PWM-WX/runs/log_wx_s1_*.txt; nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv